## Ejercicio 3 – Caso de automatización
### Preparación de datos, reglas, glosario y validaciones

En esta etapa se organiza la información del archivo `withdrawals.xlsx` para dejar lista la construcción de la lógica de decisión del punto 3.

Se cargan y validan las 4 pestañas del archivo:
- `withdrawal_requests`
- `account_snapshot`
- `pending_activity`
- `destination_registry`

Además, se dejan definidos:
- las reglas operativas del ejercicio,
- el glosario de `reason_code` y severidad,
- las aclaraciones del enunciado,
- una base enriquecida por solicitud (`request_id`),
- y tablas auxiliares para duplicados, destino reciente y actividad pendiente.

**Importante:** en esta etapa aún no se asignan decisiones finales.  
Aquí solo se deja todo preparado para construir después la lógica de `APPROVE`, `HOLD` y `REJECT`.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# EJERCICIO 3
# PREPARACION DE DATOS, REGLAS, GLOSARIO Y VALIDACIONES
# ============================================================

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    required_dirs = {"data", "docs", "notebooks", "outputs", "src"}

    for candidate in candidates:
        if candidate.exists():
            dir_names = {p.name for p in candidate.iterdir() if p.is_dir()}
            if required_dirs.issubset(dir_names):
                return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. "
        "Ejecuta este notebook dentro de ~/Documents/prueba_insights"
    )

ROOT = find_project_root()
DATA_DIR = ROOT / "data"
OUTPUTS_DIR = ROOT / "outputs"
EJ3_OUTPUT_DIR = OUTPUTS_DIR / "ejercicio_3"

EJ3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

excel_path = DATA_DIR / "withdrawals.xlsx"
if not excel_path.exists():
    raise FileNotFoundError(f"No existe el archivo: {excel_path}")

print(f"Raíz detectada: {ROOT}")
print(f"Archivo de entrada: {excel_path}")
print(f"Directorio de salida: {EJ3_OUTPUT_DIR}")

# ============================================================
# CONFIGURACION DE REGLAS DEL PUNTO 3
# ============================================================

RULES_CONFIG = {
    "BUFFER_USD": 50.0,
    "RECENT_DEST_DAYS": 7,
    "DUP_WINDOW_MIN": 15,
}

REASON_SEVERITY = {
    "KYC_NOT_VERIFIED": 100,
    "ACCOUNT_NOT_ACTIVE": 95,
    "UNWHITELISTED_HIGH_AML": 90,
    "INVALID_AMOUNT": 85,
    "DUPLICATE_REQUEST": 70,
    "INSUFFICIENT_SETTLED_AFTER_BUFFER": 65,
    "INSUFFICIENT_AVAILABLE_AFTER_BUFFER": 55,
    "DEST_CHANGED_RECENTLY": 45,
    "URGENT_RISK_TIER": 35,
}

RULES_DF = pd.DataFrame(
    [{"rule_name": k, "value": v} for k, v in RULES_CONFIG.items()]
)

REASON_GLOSSARY_DF = pd.DataFrame(
    [{"reason_code": k, "severity": v} for k, v in REASON_SEVERITY.items()]
).sort_values("severity", ascending=False).reset_index(drop=True)

INSTRUCTIONS_NOTES_DF = pd.DataFrame(
    [
        {
            "topic": "Destino reciente",
            "instruction": (
                "Comparar destination_registry.last_changed_at "
                "contra account_snapshot.as_of."
            ),
        },
        {
            "topic": "Review file",
            "instruction": (
                "Debe contener solo solicitudes HOLD y estar ordenado "
                "por severity de mayor a menor."
            ),
        },
        {
            "topic": "Prioridad lógica",
            "instruction": (
                "REJECT domina sobre HOLD y APPROVE. "
                "HOLD solo aplica si la solicitud no cae en REJECT."
            ),
        },
        {
            "topic": "Pending activity",
            "instruction": (
                "Se organiza como soporte analítico, pero no se usa "
                "directamente en las reglas explícitas del enunciado."
            ),
        },
    ]
)

# ============================================================
# CARGA DE HOJAS Y VALIDACION DE ESTRUCTURA
# ============================================================

xls = pd.ExcelFile(excel_path)
expected_sheets = {
    "withdrawal_requests",
    "account_snapshot",
    "pending_activity",
    "destination_registry",
}

found_sheets = set(xls.sheet_names)

if found_sheets != expected_sheets:
    raise ValueError(
        f"Las hojas encontradas no coinciden con las esperadas.\n"
        f"Esperadas: {sorted(expected_sheets)}\n"
        f"Encontradas: {sorted(found_sheets)}"
    )

withdrawal_requests = pd.read_excel(excel_path, sheet_name="withdrawal_requests")
account_snapshot = pd.read_excel(excel_path, sheet_name="account_snapshot")
pending_activity = pd.read_excel(excel_path, sheet_name="pending_activity")
destination_registry = pd.read_excel(excel_path, sheet_name="destination_registry")

required_columns = {
    "withdrawal_requests": [
        "request_id", "created_at", "client_id", "account_id", "amount",
        "currency", "destination_type", "destination_id", "requested_speed",
        "channel", "note"
    ],
    "account_snapshot": [
        "account_id", "client_id", "as_of", "account_status", "kyc_status",
        "aml_risk_tier", "available_cash", "settled_cash", "unsettled_cash",
        "margin_enabled", "base_currency"
    ],
    "pending_activity": [
        "account_id", "type", "status", "amount", "currency",
        "expected_settle_at", "ref_id"
    ],
    "destination_registry": [
        "destination_id", "client_id", "destination_type", "is_whitelisted",
        "last_changed_at", "country", "name_on_account"
    ],
}

loaded_dfs = {
    "withdrawal_requests": withdrawal_requests,
    "account_snapshot": account_snapshot,
    "pending_activity": pending_activity,
    "destination_registry": destination_registry,
}

for sheet_name, expected_cols in required_columns.items():
    df = loaded_dfs[sheet_name]
    missing = [c for c in expected_cols if c not in df.columns]
    extra = [c for c in df.columns if c not in expected_cols]

    if missing:
        raise ValueError(f"Faltan columnas en {sheet_name}: {missing}")

    print(f"\n{sheet_name}:")
    print(f"- shape: {df.shape}")
    print(f"- columnas esperadas presentes: OK")
    if extra:
        print(f"- columnas extra detectadas: {extra}")
    else:
        print("- columnas extra detectadas: ninguna")

# ============================================================
# NORMALIZACION DE TIPOS
# ============================================================

withdrawal_requests = withdrawal_requests.copy()
account_snapshot = account_snapshot.copy()
pending_activity = pending_activity.copy()
destination_registry = destination_registry.copy()

withdrawal_requests["created_at"] = pd.to_datetime(withdrawal_requests["created_at"], utc=True)
account_snapshot["as_of"] = pd.to_datetime(account_snapshot["as_of"], utc=True)
pending_activity["expected_settle_at"] = pd.to_datetime(pending_activity["expected_settle_at"], utc=True)
destination_registry["last_changed_at"] = pd.to_datetime(destination_registry["last_changed_at"], utc=True)

numeric_cols_requests = ["amount"]
numeric_cols_snapshot = ["available_cash", "settled_cash", "unsettled_cash"]
numeric_cols_pending = ["amount"]

for col in numeric_cols_requests:
    withdrawal_requests[col] = pd.to_numeric(withdrawal_requests[col], errors="coerce")

for col in numeric_cols_snapshot:
    account_snapshot[col] = pd.to_numeric(account_snapshot[col], errors="coerce")

for col in numeric_cols_pending:
    pending_activity[col] = pd.to_numeric(pending_activity[col], errors="coerce")

destination_registry["is_whitelisted"] = destination_registry["is_whitelisted"].astype(bool)
account_snapshot["margin_enabled"] = account_snapshot["margin_enabled"].astype(bool)

# ============================================================
# VALIDACIONES DE INTEGRIDAD
# ============================================================

if withdrawal_requests["request_id"].duplicated().any():
    duplicated_ids = withdrawal_requests.loc[
        withdrawal_requests["request_id"].duplicated(), "request_id"
    ].tolist()
    raise ValueError(f"request_id duplicados encontrados: {duplicated_ids[:10]}")

if account_snapshot["account_id"].duplicated().any():
    duplicated_accounts = account_snapshot.loc[
        account_snapshot["account_id"].duplicated(), "account_id"
    ].tolist()
    raise ValueError(f"account_id duplicados en account_snapshot: {duplicated_accounts[:10]}")

if destination_registry["destination_id"].duplicated().any():
    duplicated_destinations = destination_registry.loc[
        destination_registry["destination_id"].duplicated(), "destination_id"
    ].tolist()
    raise ValueError(f"destination_id duplicados en destination_registry: {duplicated_destinations[:10]}")

missing_accounts = withdrawal_requests.loc[
    ~withdrawal_requests["account_id"].isin(account_snapshot["account_id"]),
    "account_id"
].unique()

missing_destinations = withdrawal_requests.loc[
    ~withdrawal_requests["destination_id"].isin(destination_registry["destination_id"]),
    "destination_id"
].unique()

if len(missing_accounts) > 0:
    raise ValueError(f"Hay account_id en withdrawal_requests que no están en account_snapshot: {missing_accounts}")

if len(missing_destinations) > 0:
    raise ValueError(
        f"Hay destination_id en withdrawal_requests que no están en destination_registry: {missing_destinations}"
    )

# Validación de consistencia de client_id por account_id
request_snapshot_check = withdrawal_requests.merge(
    account_snapshot[["account_id", "client_id"]],
    on="account_id",
    how="left",
    suffixes=("_request", "_snapshot"),
)

client_mismatch_snapshot = request_snapshot_check.loc[
    request_snapshot_check["client_id_request"] != request_snapshot_check["client_id_snapshot"]
]

if not client_mismatch_snapshot.empty:
    raise ValueError(
        "Hay inconsistencias entre client_id de withdrawal_requests y account_snapshot."
    )

# Validación de consistencia de client_id por destination_id
request_destination_check = withdrawal_requests.merge(
    destination_registry[["destination_id", "client_id"]],
    on="destination_id",
    how="left",
    suffixes=("_request", "_destination"),
)

client_mismatch_destination = request_destination_check.loc[
    request_destination_check["client_id_request"] != request_destination_check["client_id_destination"]
]

if not client_mismatch_destination.empty:
    raise ValueError(
        "Hay inconsistencias entre client_id de withdrawal_requests y destination_registry."
    )

print("\nValidaciones de integridad:")
print("- request_id únicos: OK")
print("- account_id y destination_id referencian correctamente: OK")
print("- client_id consistente entre tablas: OK")

# ============================================================
# RENOMBRE DE COLUMNAS PARA JOIN LIMPIO
# ============================================================

account_snapshot_prepared = account_snapshot.rename(
    columns={
        "as_of": "snapshot_as_of",
        "client_id": "snapshot_client_id",
        "account_status": "snapshot_account_status",
        "kyc_status": "snapshot_kyc_status",
        "aml_risk_tier": "snapshot_aml_risk_tier",
        "available_cash": "snapshot_available_cash",
        "settled_cash": "snapshot_settled_cash",
        "unsettled_cash": "snapshot_unsettled_cash",
        "margin_enabled": "snapshot_margin_enabled",
        "base_currency": "snapshot_base_currency",
    }
)

destination_registry_prepared = destination_registry.rename(
    columns={
        "client_id": "destination_client_id",
        "destination_type": "registry_destination_type",
        "is_whitelisted": "dest_is_whitelisted",
        "last_changed_at": "dest_last_changed_at",
        "country": "dest_country",
        "name_on_account": "dest_name_on_account",
    }
)

# ============================================================
# RESUMEN DE PENDING_ACTIVITY
# ============================================================
# Nota:
# pending_activity no entra de forma explícita en las reglas del PDF,
# pero se deja organizada por cuenta como soporte analítico.

pending_activity_prepared = pending_activity.copy()
pending_activity_prepared["status_norm"] = pending_activity_prepared["status"].astype(str).str.strip().str.lower()
pending_activity_prepared["type_norm"] = pending_activity_prepared["type"].astype(str).str.strip().str.lower()

pending_open = pending_activity_prepared.loc[
    pending_activity_prepared["status_norm"] == "pending"
].copy()

if pending_open.empty:
    pending_activity_summary = pd.DataFrame(
        columns=[
            "account_id",
            "pending_items_count",
            "pending_amount_net",
            "pending_amount_abs",
            "next_expected_settle_at",
        ]
    )
else:
    pending_activity_summary = (
        pending_open
        .groupby("account_id", as_index=False)
        .agg(
            pending_items_count=("ref_id", "count"),
            pending_amount_net=("amount", "sum"),
            pending_amount_abs=("amount", lambda s: s.abs().sum()),
            next_expected_settle_at=("expected_settle_at", "min"),
        )
    )

# ============================================================
# DETECCION DE DUPLICADOS CANDIDATOS
# Regla del enunciado:
# mismo account_id + amount + destination_id dentro de 15 min
# ============================================================

dup_keys = ["account_id", "amount", "destination_id"]

duplicates_helper = withdrawal_requests.copy().sort_values(
    dup_keys + ["created_at"]
).reset_index(drop=True)

grouped = duplicates_helper.groupby(dup_keys, dropna=False)

duplicates_helper["prev_created_at"] = grouped["created_at"].shift(1)
duplicates_helper["prev_request_id"] = grouped["request_id"].shift(1)

duplicates_helper["next_created_at"] = grouped["created_at"].shift(-1)
duplicates_helper["next_request_id"] = grouped["request_id"].shift(-1)

duplicates_helper["mins_from_prev"] = (
    (duplicates_helper["created_at"] - duplicates_helper["prev_created_at"])
    .dt.total_seconds()
    .div(60)
)

duplicates_helper["mins_to_next"] = (
    (duplicates_helper["next_created_at"] - duplicates_helper["created_at"])
    .dt.total_seconds()
    .div(60)
)

duplicates_helper["dup_with_prev_15m"] = (
    duplicates_helper["mins_from_prev"].notna()
    & (duplicates_helper["mins_from_prev"] <= RULES_CONFIG["DUP_WINDOW_MIN"])
)

duplicates_helper["dup_with_next_15m"] = (
    duplicates_helper["mins_to_next"].notna()
    & (duplicates_helper["mins_to_next"] <= RULES_CONFIG["DUP_WINDOW_MIN"])
)

duplicates_helper["duplicate_candidate_flag"] = (
    duplicates_helper["dup_with_prev_15m"] | duplicates_helper["dup_with_next_15m"]
)

duplicates_helper["duplicate_pair_request_id"] = np.where(
    duplicates_helper["dup_with_prev_15m"],
    duplicates_helper["prev_request_id"],
    np.where(
        duplicates_helper["dup_with_next_15m"],
        duplicates_helper["next_request_id"],
        pd.NA
    )
)

duplicate_candidates = duplicates_helper.loc[
    duplicates_helper["duplicate_candidate_flag"]
].copy()

# ============================================================
# BASE ENRIQUECIDA POR SOLICITUD
# ============================================================

requests_enriched_base = (
    withdrawal_requests
    .merge(
        account_snapshot_prepared,
        left_on="account_id",
        right_on="account_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        destination_registry_prepared,
        left_on="destination_id",
        right_on="destination_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        pending_activity_summary,
        on="account_id",
        how="left",
        validate="many_to_one",
    )
)

# ============================================================
# HELPERS CALCULADOS PARA LA LOGICA DEL PUNTO 3
# ============================================================

requests_enriched_base["available_cash_after_request"] = (
    requests_enriched_base["snapshot_available_cash"] - requests_enriched_base["amount"]
)

requests_enriched_base["settled_cash_after_request"] = (
    requests_enriched_base["snapshot_settled_cash"] - requests_enriched_base["amount"]
)

requests_enriched_base["dest_days_since_change"] = (
    (requests_enriched_base["snapshot_as_of"] - requests_enriched_base["dest_last_changed_at"])
    .dt.total_seconds()
    .div(86400)
)

requests_enriched_base["dest_changed_recently_flag"] = (
    requests_enriched_base["dest_days_since_change"] < RULES_CONFIG["RECENT_DEST_DAYS"]
)

requests_enriched_base["urgent_risk_flag"] = (
    requests_enriched_base["requested_speed"].astype(str).str.lower().eq("urgent")
    & requests_enriched_base["snapshot_aml_risk_tier"].isin(["medium", "high"])
)

duplicate_flags_small = duplicate_candidates[[
    "request_id",
    "duplicate_candidate_flag",
    "duplicate_pair_request_id",
    "mins_from_prev",
    "mins_to_next",
]]

requests_enriched_base = requests_enriched_base.merge(
    duplicate_flags_small,
    on="request_id",
    how="left",
)

requests_enriched_base["duplicate_candidate_flag"] = (
    requests_enriched_base["duplicate_candidate_flag"].fillna(False)
)

# Orden sugerido de columnas principales
preferred_cols = [
    "request_id",
    "created_at",
    "client_id",
    "account_id",
    "amount",
    "currency",
    "destination_type",
    "destination_id",
    "requested_speed",
    "channel",
    "note",
    "snapshot_as_of",
    "snapshot_account_status",
    "snapshot_kyc_status",
    "snapshot_aml_risk_tier",
    "snapshot_available_cash",
    "snapshot_settled_cash",
    "snapshot_unsettled_cash",
    "snapshot_margin_enabled",
    "snapshot_base_currency",
    "dest_is_whitelisted",
    "dest_last_changed_at",
    "dest_country",
    "dest_name_on_account",
    "pending_items_count",
    "pending_amount_net",
    "pending_amount_abs",
    "next_expected_settle_at",
    "available_cash_after_request",
    "settled_cash_after_request",
    "dest_days_since_change",
    "dest_changed_recently_flag",
    "urgent_risk_flag",
    "duplicate_candidate_flag",
    "duplicate_pair_request_id",
    "mins_from_prev",
    "mins_to_next",
]

preferred_cols = [c for c in preferred_cols if c in requests_enriched_base.columns]
requests_enriched_base = requests_enriched_base[preferred_cols].copy()

# ============================================================
# TABLAS RESUMEN PARA EL NOTEBOOK
# ============================================================

data_dictionary_df = pd.DataFrame([
    {
        "table_name": "withdrawal_requests",
        "purpose": "Solicitudes de retiro; tabla base por request_id.",
        "rows": len(withdrawal_requests),
        "columns": withdrawal_requests.shape[1],
    },
    {
        "table_name": "account_snapshot",
        "purpose": "Estado de la cuenta al corte; fuente de cash, KYC, AML y status.",
        "rows": len(account_snapshot),
        "columns": account_snapshot.shape[1],
    },
    {
        "table_name": "pending_activity",
        "purpose": "Actividad pendiente por cuenta; soporte analítico.",
        "rows": len(pending_activity),
        "columns": pending_activity.shape[1],
    },
    {
        "table_name": "destination_registry",
        "purpose": "Registro de destinos; fuente de whitelist y cambio reciente.",
        "rows": len(destination_registry),
        "columns": destination_registry.shape[1],
    },
    {
        "table_name": "requests_enriched_base",
        "purpose": "Base final organizada para construir la lógica de decisión.",
        "rows": len(requests_enriched_base),
        "columns": requests_enriched_base.shape[1],
    },
])

# ============================================================
# MOSTRAR RESUMEN EN NOTEBOOK
# ============================================================

print("\nResumen de tablas:")
display(data_dictionary_df)

print("\nReglas de configuración:")
display(RULES_DF)

print("\nGlosario de reason_code y severidad:")
display(REASON_GLOSSARY_DF)

print("\nAclaraciones relevantes del punto 3:")
display(INSTRUCTIONS_NOTES_DF)

print("\nVista previa de pending_activity_summary:")
display(pending_activity_summary.head(10))

print("\nVista previa de duplicate_candidates:")
display(duplicate_candidates.head(10))

print("\nVista previa de requests_enriched_base:")
display(requests_enriched_base.head(10))

# ============================================================
# EXPORTAR RESULTADOS ORGANIZADOS
# ============================================================

RULES_DF.to_csv(EJ3_OUTPUT_DIR / "ejercicio_3_rules_config.csv", index=False)
REASON_GLOSSARY_DF.to_csv(EJ3_OUTPUT_DIR / "ejercicio_3_reason_glossary.csv", index=False)
INSTRUCTIONS_NOTES_DF.to_csv(EJ3_OUTPUT_DIR / "ejercicio_3_instruction_notes.csv", index=False)

withdrawal_requests.to_csv(EJ3_OUTPUT_DIR / "ejercicio_3_withdrawal_requests_clean.csv", index=False)
account_snapshot_prepared.to_csv(EJ3_OUTPUT_DIR / "ejercicio_3_account_snapshot_clean.csv", index=False)
pending_activity_prepared.to_csv(EJ3_OUTPUT_DIR / "ejercicio_3_pending_activity_clean.csv", index=False)
destination_registry_prepared.to_csv(EJ3_OUTPUT_DIR / "ejercicio_3_destination_registry_clean.csv", index=False)

pending_activity_summary.to_csv(EJ3_OUTPUT_DIR / "ejercicio_3_pending_activity_summary.csv", index=False)
duplicate_candidates.to_csv(EJ3_OUTPUT_DIR / "ejercicio_3_duplicate_candidates.csv", index=False)
requests_enriched_base.to_csv(EJ3_OUTPUT_DIR / "ejercicio_3_requests_enriched_base.csv", index=False)

print("\nArchivos generados:")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_rules_config.csv'}")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_reason_glossary.csv'}")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_instruction_notes.csv'}")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_withdrawal_requests_clean.csv'}")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_account_snapshot_clean.csv'}")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_pending_activity_clean.csv'}")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_destination_registry_clean.csv'}")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_pending_activity_summary.csv'}")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_duplicate_candidates.csv'}")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_requests_enriched_base.csv'}")

Raíz detectada: /Users/santiago.os/Documents/prueba_insights
Archivo de entrada: /Users/santiago.os/Documents/prueba_insights/data/withdrawals.xlsx
Directorio de salida: /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3

withdrawal_requests:
- shape: (128, 11)
- columnas esperadas presentes: OK
- columnas extra detectadas: ninguna

account_snapshot:
- shape: (35, 11)
- columnas esperadas presentes: OK
- columnas extra detectadas: ninguna

pending_activity:
- shape: (99, 7)
- columnas esperadas presentes: OK
- columnas extra detectadas: ninguna

destination_registry:
- shape: (52, 7)
- columnas esperadas presentes: OK
- columnas extra detectadas: ninguna

Validaciones de integridad:
- request_id únicos: OK
- account_id y destination_id referencian correctamente: OK
- client_id consistente entre tablas: OK

Resumen de tablas:


,table_name,purpose,rows,columns
0,withdrawal_requests,Solicitudes de retiro; tabla base por request_id.,128,11
1,account_snapshot,"Estado de la cuenta al corte; fuente de cash, ...",35,11
2,pending_activity,Actividad pendiente por cuenta; soporte analít...,99,7
3,destination_registry,Registro de destinos; fuente de whitelist y ca...,52,7
4,requests_enriched_base,Base final organizada para construir la lógica...,128,37



Reglas de configuración:


,rule_name,value
0,BUFFER_USD,50.0
1,RECENT_DEST_DAYS,7.0
2,DUP_WINDOW_MIN,15.0



Glosario de reason_code y severidad:


...


Vista previa de duplicate_candidates:


,request_id,created_at,client_id,account_id,amount,currency,destination_type,destination_id,requested_speed,channel,...,prev_created_at,prev_request_id,next_created_at,next_request_id,mins_from_prev,mins_to_next,dup_with_prev_15m,dup_with_next_15m,duplicate_candidate_flag,duplicate_pair_request_id
11,W100044,2026-03-05 09:08:16.872327+00:00,C020,A0003,10244.98,USD,bank,D00036,standard,advisor,...,NaT,NaN,2026-03-05 09:17:16.872327+00:00,W100044_DUP,NaN,9.0,False,True,True,W100044_DUP
12,W100044_DUP,2026-03-05 09:17:16.872327+00:00,C020,A0003,10244.98,USD,bank,D00036,standard,advisor,...,2026-03-05 09:08:16.872327+00:00,W100044,NaT,NaN,9.0,NaN,True,False,True,W100044
69,W100026,2026-03-06 08:31:21.341604+00:00,C016,A0021,1145.83,USD,bank,D00026,standard,app,...,NaT,NaN,2026-03-06 08:42:21.341604+00:00,W100026_DUP,NaN,11.0,False,True,True,W100026_DUP
70,W100026_DUP,2026-03-06 08:42:21.341604+00:00,C016,A0021,1145.83,USD,bank,D00026,standard,app,...,2026-03-06 08:31:21.341604+00:00,W100026,NaT,NaN,11.0,NaN,True,False,True,W100026
84,W100073,2026-03-05 17:49:45.791614+00:00,C024,A0025,3063.75,USD,bank,D00040,standard,app,...,NaT,NaN,2026-03-05 17:55:45.791614+00:00,W100073_DUP,NaN,6.0,False,True,True,W100073_DUP
85,W100073_DUP,2026-03-05 17:55:45.791614+00:00,C024,A0025,3063.75,USD,bank,D00040,standard,app,...,2026-03-05 17:49:45.791614+00:00,W100073,NaT,NaN,6.0,NaN,True,False,True,W100073
89,W100004,2026-03-05 09:19:55.351331+00:00,C020,A0026,21878.31,USD,bank,D00036,standard,app,...,NaT,NaN,2026-03-05 09:30:55.351331+00:00,W100004_DUP,NaN,11.0,False,True,True,W100004_DUP
90,W100004_DUP,2026-03-05 09:30:55.351331+00:00,C020,A0026,21878.31,USD,bank,D00036,standard,app,...,2026-03-05 09:19:55.351331+00:00,W100004,NaT,NaN,11.0,NaN,True,False,True,W100004
92,W100047,2026-03-06 13:01:45.706405+00:00,C013,A0027,91705.36,USD,bank,D00023,standard,app,...,NaT,NaN,2026-03-06 13:11:45.706405+00:00,W100047_DUP,NaN,10.0,False,True,True,W100047_DUP
93,W100047_DUP,2026-03-06 13:11:45.706405+00:00,C013,A0027,91705.36,USD,bank,D00023,standard,app,...,2026-03-06 13:01:45.706405+00:00,W100047,NaT,NaN,10.0,NaN,True,False,True,W100047



Vista previa de requests_enriched_base:


,request_id,created_at,client_id,account_id,amount,currency,destination_type,destination_id,requested_speed,channel,...,next_expected_settle_at,available_cash_after_request,settled_cash_after_request,dest_days_since_change,dest_changed_recently_flag,urgent_risk_flag,duplicate_candidate_flag,duplicate_pair_request_id,mins_from_prev,mins_to_next
0,W100055,2026-03-06 01:00:13.581307+00:00,C007,A0032,2000.27,USD,crypto,D00012,standard,app,...,NaT,10529.10,9684.51,252.250000,False,False,True,W100055_DUP,NaN,7.0
1,W100040,2026-03-06 04:27:02.907998+00:00,C021,A0008,4048.65,USD,bank,D00037,standard,app,...,2026-03-04 01:00:00+00:00,16395.72,12207.14,336.958333,False,False,False,NaN,NaN,NaN
2,W100019,2026-03-06 01:37:44.488632+00:00,C023,A0014,3352.79,USD,bank,D00039,standard,ops,...,2026-03-03 18:00:00+00:00,12462.44,12462.44,2.166667,True,False,False,NaN,NaN,NaN
3,W100031,2026-03-05 15:24:11.787265+00:00,C013,A0027,26978.71,USD,bank,D00023,standard,ops,...,NaT,87638.41,64509.17,99.625000,False,False,False,NaN,NaN,NaN
4,W100098,2026-03-05 11:47:49.487155+00:00,C006,A0023,2861.14,USD,bank,D00011,standard,app,...,2026-03-05 17:00:00+00:00,37091.61,30407.72,170.000000,False,False,False,NaN,NaN,NaN
5,W100056,2026-03-06 05:06:32.601379+00:00,C003,A0033,2707.10,USD,bank,D00005,standard,advisor,...,2026-03-06 10:00:00+00:00,4873.03,2528.05,82.666667,False,False,False,NaN,NaN,NaN
6,W100069,2026-03-05 21:45:21.294399+00:00,C014,A0020,8906.77,USD,bank,D00024,standard,app,...,2026-03-06 06:00:00+00:00,-481.46,-481.46,60.125000,False,False,False,NaN,NaN,NaN
7,W100104,2026-03-06 03:12:11.042107+00:00,C017,A0034,7788.04,USD,bank,D00029,standard,advisor,...,2026-03-08 19:00:00+00:00,35289.95,32074.43,195.333333,False,False,False,NaN,NaN,NaN
8,W100081,2026-03-06 09:58:40.891828+00:00,C020,A0003,4398.52,USD,bank,D00036,urgent,advisor,...,NaT,22236.28,22236.28,4.666667,True,False,False,NaN,NaN,NaN
9,W100026,2026-03-06 08:31:21.341604+00:00,C016,A0021,1145.83,USD,bank,D00026,standard,app,...,NaT,1483.73,1483.73,2.291667,True,False,True,W100026_DUP,NaN,11.0



Archivos generados:
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_rules_config.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_reason_glossary.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_instruction_notes.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_withdrawal_requests_clean.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_account_snapshot_clean.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_pending_activity_clean.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_destination_registry_clean.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_pending_activity_summary.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_duplicate_candidates.csv
- /Users/santiago.os/Documents/prueba_in

## Ejercicio 3 – Motor de decisión
### Asignación de `APPROVE`, `HOLD`, `REJECT`, `reason_code` y `severity`

En esta etapa se aplica la lógica de negocio sobre la base previamente enriquecida por solicitud (`requests_enriched_base`).

La asignación sigue estas reglas:

### Reglas de rechazo (`REJECT`)
- `KYC_NOT_VERIFIED`
- `ACCOUNT_NOT_ACTIVE`
- `UNWHITELISTED_HIGH_AML`
- `INVALID_AMOUNT`

### Reglas de retención manual (`HOLD`)
- `DUPLICATE_REQUEST`
- `INSUFFICIENT_SETTLED_AFTER_BUFFER`
- `INSUFFICIENT_AVAILABLE_AFTER_BUFFER`
- `DEST_CHANGED_RECENTLY`
- `URGENT_RISK_TIER`

### Regla de prioridad
- `REJECT` domina sobre `HOLD`
- `HOLD` domina sobre `APPROVE`
- cuando una solicitud activa múltiples reglas, se asigna el `reason_code` de mayor severidad

Además, se genera un archivo de revisión con solo solicitudes `HOLD`, ordenado por severidad de mayor a menor.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# EJERCICIO 3
# MOTOR DE DECISION
# ============================================================

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    required_dirs = {"data", "docs", "notebooks", "outputs", "src"}

    for candidate in candidates:
        if candidate.exists():
            dir_names = {p.name for p in candidate.iterdir() if p.is_dir()}
            if required_dirs.issubset(dir_names):
                return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. "
        "Ejecuta este notebook dentro de ~/Documents/prueba_insights"
    )

ROOT = find_project_root()
OUTPUTS_DIR = ROOT / "outputs"
EJ3_OUTPUT_DIR = OUTPUTS_DIR / "ejercicio_3"

base_path = EJ3_OUTPUT_DIR / "ejercicio_3_requests_enriched_base.csv"
rules_path = EJ3_OUTPUT_DIR / "ejercicio_3_rules_config.csv"
glossary_path = EJ3_OUTPUT_DIR / "ejercicio_3_reason_glossary.csv"

for path in [base_path, rules_path, glossary_path]:
    if not path.exists():
        raise FileNotFoundError(f"No existe el archivo requerido: {path}")

print(f"Base cargada desde: {base_path}")
print(f"Rules cargadas desde: {rules_path}")
print(f"Glossary cargado desde: {glossary_path}")

# ============================================================
# CARGA DE INSUMOS PREPARADOS
# ============================================================

base = pd.read_csv(base_path)
rules_df = pd.read_csv(rules_path)
glossary_df = pd.read_csv(glossary_path)

# ============================================================
# NORMALIZACION DE TIPOS
# ============================================================

datetime_cols = [
    "created_at",
    "snapshot_as_of",
    "dest_last_changed_at",
    "next_expected_settle_at",
]

for col in datetime_cols:
    if col in base.columns:
        base[col] = pd.to_datetime(base[col], errors="coerce", utc=True)

bool_cols = [
    "dest_is_whitelisted",
    "snapshot_margin_enabled",
    "dest_changed_recently_flag",
    "urgent_risk_flag",
    "duplicate_candidate_flag",
]

for col in bool_cols:
    if col in base.columns:
        if base[col].dtype == object:
            base[col] = (
                base[col]
                .astype(str)
                .str.strip()
                .str.lower()
                .map({"true": True, "false": False})
            )

numeric_cols = [
    "amount",
    "snapshot_available_cash",
    "snapshot_settled_cash",
    "snapshot_unsettled_cash",
    "pending_items_count",
    "pending_amount_net",
    "pending_amount_abs",
    "available_cash_after_request",
    "settled_cash_after_request",
    "dest_days_since_change",
    "mins_from_prev",
    "mins_to_next",
]

for col in numeric_cols:
    if col in base.columns:
        base[col] = pd.to_numeric(base[col], errors="coerce")

# ============================================================
# CARGA DE PARAMETROS Y GLOSARIO
# ============================================================

RULES_CONFIG = dict(zip(rules_df["rule_name"], rules_df["value"]))
REASON_SEVERITY = dict(zip(glossary_df["reason_code"], glossary_df["severity"]))

BUFFER_USD = float(RULES_CONFIG["BUFFER_USD"])

REJECT_REASON_CODES = {
    "KYC_NOT_VERIFIED",
    "ACCOUNT_NOT_ACTIVE",
    "UNWHITELISTED_HIGH_AML",
    "INVALID_AMOUNT",
}

HOLD_REASON_CODES = {
    "DUPLICATE_REQUEST",
    "INSUFFICIENT_SETTLED_AFTER_BUFFER",
    "INSUFFICIENT_AVAILABLE_AFTER_BUFFER",
    "DEST_CHANGED_RECENTLY",
    "URGENT_RISK_TIER",
}

# ============================================================
# NORMALIZACION DE TEXTO
# ============================================================

def normalize_text_series(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.lower()

base["snapshot_kyc_status_norm"] = normalize_text_series(base["snapshot_kyc_status"])
base["snapshot_account_status_norm"] = normalize_text_series(base["snapshot_account_status"])
base["snapshot_aml_risk_tier_norm"] = normalize_text_series(base["snapshot_aml_risk_tier"])
base["requested_speed_norm"] = normalize_text_series(base["requested_speed"])

# ============================================================
# FLAGS DE REGLAS
# Nota: se mantiene la interpretacion conservadora de duplicados:
# si una solicitud participa en un par duplicado, se marca para HOLD.
# ============================================================

rule_flags = {
    "KYC_NOT_VERIFIED": (
        base["snapshot_kyc_status_norm"] != "verified"
    ),

    "ACCOUNT_NOT_ACTIVE": (
        base["snapshot_account_status_norm"] != "active"
    ),

    "UNWHITELISTED_HIGH_AML": (
        (base["snapshot_aml_risk_tier_norm"] == "high")
        & (~base["dest_is_whitelisted"].fillna(False))
    ),

    "INVALID_AMOUNT": (
        base["amount"].isna() | (base["amount"] <= 0)
    ),

    "DUPLICATE_REQUEST": (
        base["duplicate_candidate_flag"].fillna(False)
    ),

    "INSUFFICIENT_SETTLED_AFTER_BUFFER": (
        base["settled_cash_after_request"] < BUFFER_USD
    ),

    "INSUFFICIENT_AVAILABLE_AFTER_BUFFER": (
        base["available_cash_after_request"] < BUFFER_USD
    ),

    "DEST_CHANGED_RECENTLY": (
        base["dest_changed_recently_flag"].fillna(False)
    ),

    "URGENT_RISK_TIER": (
        base["urgent_risk_flag"].fillna(False)
    ),
}

for reason_code, flag_series in rule_flags.items():
    base[f"flag_{reason_code}"] = flag_series.astype(bool)

# ============================================================
# RESUMEN DE ACTIVACION DE REGLAS
# ============================================================

rule_hits_df = pd.DataFrame(
    [
        {
            "reason_code": reason_code,
            "severity": REASON_SEVERITY[reason_code],
            "hit_count": int(base[f"flag_{reason_code}"].sum()),
        }
        for reason_code in REASON_SEVERITY.keys()
    ]
).sort_values(["severity", "reason_code"], ascending=[False, True]).reset_index(drop=True)

# ============================================================
# FUNCION DE DECISION POR FILA
# ============================================================

def evaluate_request(row: pd.Series) -> pd.Series:
    triggered_rules = [
        reason_code
        for reason_code in REASON_SEVERITY.keys()
        if bool(row[f"flag_{reason_code}"])
    ]

    if not triggered_rules:
        return pd.Series(
            {
                "decision": "APPROVE",
                "reason_code": pd.NA,
                "severity": pd.NA,
                "triggered_rules": pd.NA,
            }
        )

    # Se toma la regla disparada de mayor severidad
    top_reason = max(triggered_rules, key=lambda x: REASON_SEVERITY[x])
    top_severity = REASON_SEVERITY[top_reason]

    if top_reason in REJECT_REASON_CODES:
        decision = "REJECT"
    elif top_reason in HOLD_REASON_CODES:
        decision = "HOLD"
    else:
        raise ValueError(f"reason_code no clasificado: {top_reason}")

    return pd.Series(
        {
            "decision": decision,
            "reason_code": top_reason,
            "severity": top_severity,
            "triggered_rules": " | ".join(triggered_rules),
        }
    )

decision_cols = base.apply(evaluate_request, axis=1)
decisions = pd.concat([base.copy(), decision_cols], axis=1)

# ============================================================
# VALIDACIONES DE CONSISTENCIA
# ============================================================

valid_decisions = {"APPROVE", "HOLD", "REJECT"}
if not set(decisions["decision"].dropna().unique()).issubset(valid_decisions):
    raise ValueError("Se detectaron decisiones inválidas.")

invalid_reason_codes = decisions.loc[
    decisions["reason_code"].notna()
    & ~decisions["reason_code"].isin(REASON_SEVERITY.keys()),
    "reason_code"
].unique()

if len(invalid_reason_codes) > 0:
    raise ValueError(f"Se detectaron reason_code inválidos: {invalid_reason_codes}")

severity_check = decisions.loc[decisions["reason_code"].notna(), ["reason_code", "severity"]].copy()
severity_check["expected_severity"] = severity_check["reason_code"].map(REASON_SEVERITY)

if not (severity_check["severity"] == severity_check["expected_severity"]).all():
    raise ValueError("Hay inconsistencias entre severity y el glosario.")

# ============================================================
# TABLA FINAL DE SALIDA
# ============================================================

final_decisions = decisions[
    [
        "request_id",
        "decision",
        "reason_code",
        "severity",
        "created_at",
        "client_id",
        "account_id",
        "amount",
        "currency",
        "destination_type",
        "destination_id",
        "requested_speed",
        "channel",
        "snapshot_account_status",
        "snapshot_kyc_status",
        "snapshot_aml_risk_tier",
        "snapshot_available_cash",
        "snapshot_settled_cash",
        "snapshot_unsettled_cash",
        "available_cash_after_request",
        "settled_cash_after_request",
        "dest_is_whitelisted",
        "dest_days_since_change",
        "dest_changed_recently_flag",
        "urgent_risk_flag",
        "duplicate_candidate_flag",
        "duplicate_pair_request_id",
        "mins_from_prev",
        "mins_to_next",
        "triggered_rules",
    ]
].copy()

decision_order_map = {"REJECT": 0, "HOLD": 1, "APPROVE": 2}
final_decisions["decision_order"] = final_decisions["decision"].map(decision_order_map)

final_decisions = final_decisions.sort_values(
    ["decision_order", "severity", "created_at", "request_id"],
    ascending=[True, False, True, True]
).drop(columns="decision_order").reset_index(drop=True)

# ============================================================
# ARCHIVO DE REVIEW: SOLO HOLD, ORDENADO POR SEVERIDAD DESC
# ============================================================

review_queue = final_decisions.loc[
    final_decisions["decision"] == "HOLD"
].copy()

review_queue = review_queue.sort_values(
    ["severity", "created_at", "request_id"],
    ascending=[False, True, True]
).reset_index(drop=True)

if not review_queue["decision"].eq("HOLD").all():
    raise ValueError("El review_queue contiene decisiones distintas de HOLD.")

# ============================================================
# TABLAS RESUMEN
# ============================================================

decision_summary = (
    final_decisions
    .groupby(["decision", "reason_code"], dropna=False, as_index=False)
    .size()
    .rename(columns={"size": "request_count"})
    .sort_values(["decision", "request_count", "reason_code"], ascending=[True, False, True])
    .reset_index(drop=True)
)

decision_counts = (
    final_decisions["decision"]
    .value_counts(dropna=False)
    .rename_axis("decision")
    .reset_index(name="request_count")
)

# ============================================================
# MOSTRAR RESULTADOS EN NOTEBOOK
# ============================================================

print("\nResumen de activación de reglas:")
display(rule_hits_df)

print("\nConteo de decisiones:")
display(decision_counts)

print("\nResumen por decision y reason_code:")
display(decision_summary)

print("\nVista previa de final_decisions:")
display(final_decisions.head(15))

print("\nVista previa de review_queue (solo HOLD):")
display(review_queue.head(15))

# ============================================================
# EXPORTACION DE RESULTADOS
# ============================================================

rule_hits_df.to_csv(EJ3_OUTPUT_DIR / "ejercicio_3_rule_hits.csv", index=False)
final_decisions.to_csv(EJ3_OUTPUT_DIR / "ejercicio_3_final_decisions.csv", index=False)
review_queue.to_csv(EJ3_OUTPUT_DIR / "ejercicio_3_review_queue.csv", index=False)
decision_summary.to_csv(EJ3_OUTPUT_DIR / "ejercicio_3_decision_summary.csv", index=False)
decision_counts.to_csv(EJ3_OUTPUT_DIR / "ejercicio_3_decision_counts.csv", index=False)

print("\nArchivos generados:")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_rule_hits.csv'}")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_final_decisions.csv'}")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_review_queue.csv'}")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_decision_summary.csv'}")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_decision_counts.csv'}")

Base cargada desde: /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_requests_enriched_base.csv
Rules cargadas desde: /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_rules_config.csv
Glossary cargado desde: /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_reason_glossary.csv

Resumen de activación de reglas:


,reason_code,severity,hit_count
0,KYC_NOT_VERIFIED,100,7
1,ACCOUNT_NOT_ACTIVE,95,20
2,UNWHITELISTED_HIGH_AML,90,0
3,INVALID_AMOUNT,85,1
4,DUPLICATE_REQUEST,70,16
5,INSUFFICIENT_SETTLED_AFTER_BUFFER,65,18
6,INSUFFICIENT_AVAILABLE_AFTER_BUFFER,55,5
7,DEST_CHANGED_RECENTLY,45,25
8,URGENT_RISK_TIER,35,18



Conteo de decisiones:


,decision,request_count
0,APPROVE,51
1,HOLD,49
2,REJECT,28



Resumen por decision y reason_code:


...


Vista previa de final_decisions:


,request_id,decision,reason_code,severity,created_at,client_id,account_id,amount,currency,destination_type,...,settled_cash_after_request,dest_is_whitelisted,dest_days_since_change,dest_changed_recently_flag,urgent_risk_flag,duplicate_candidate_flag,duplicate_pair_request_id,mins_from_prev,mins_to_next,triggered_rules
0,W100105,REJECT,KYC_NOT_VERIFIED,100.0,2026-03-05 04:50:18.568126+00:00,C012,A0022,992.26,USD,bank,...,2583.02,True,171.416667,False,True,False,NaN,NaN,NaN,KYC_NOT_VERIFIED | URGENT_RISK_TIER
1,W100082,REJECT,KYC_NOT_VERIFIED,100.0,2026-03-05 08:00:06.870504+00:00,C030,A0012,4593.63,USD,bank,...,13627.04,True,131.958333,False,False,False,NaN,NaN,NaN,KYC_NOT_VERIFIED
2,W100001,REJECT,KYC_NOT_VERIFIED,100.0,2026-03-05 11:04:06.474179+00:00,C024,A0016,3049.90,USD,bank,...,4158.37,True,23.000000,False,False,False,NaN,NaN,NaN,KYC_NOT_VERIFIED
3,W100051,REJECT,KYC_NOT_VERIFIED,100.0,2026-03-05 23:48:43.603467+00:00,C030,A0012,18378.93,USD,bank,...,-158.26,True,131.958333,False,False,False,NaN,NaN,NaN,KYC_NOT_VERIFIED | INSUFFICIENT_SETTLED_AFTER_...
4,W100087,REJECT,KYC_NOT_VERIFIED,100.0,2026-03-06 05:19:11.650987+00:00,C023,A0013,5836.91,USD,bank,...,7369.03,True,2.166667,True,False,False,NaN,NaN,NaN,KYC_NOT_VERIFIED | DEST_CHANGED_RECENTLY
5,W100050,REJECT,KYC_NOT_VERIFIED,100.0,2026-03-06 05:29:07.401619+00:00,C024,A0016,2844.67,USD,bank,...,4363.60,True,23.000000,False,False,False,NaN,NaN,NaN,KYC_NOT_VERIFIED
6,W100020,REJECT,KYC_NOT_VERIFIED,100.0,2026-03-06 06:15:36.687420+00:00,C030,A0012,5257.29,USD,bank,...,12963.38,True,131.958333,False,True,False,NaN,NaN,NaN,KYC_NOT_VERIFIED | URGENT_RISK_TIER
7,W100070,REJECT,ACCOUNT_NOT_ACTIVE,95.0,2026-03-05 05:18:27.354278+00:00,C003,A0001,16280.18,USD,bank,...,-194.29,True,82.666667,False,False,False,NaN,NaN,NaN,ACCOUNT_NOT_ACTIVE | INSUFFICIENT_SETTLED_AFTE...
8,W100067,REJECT,ACCOUNT_NOT_ACTIVE,95.0,2026-03-05 06:55:42.780600+00:00,C028,A0024,913.54,USD,crypto,...,1986.78,True,258.875000,False,True,False,NaN,NaN,NaN,ACCOUNT_NOT_ACTIVE | URGENT_RISK_TIER
9,W100017,REJECT,ACCOUNT_NOT_ACTIVE,95.0,2026-03-05 11:56:25.719473+00:00,C003,A0001,4785.33,USD,bank,...,11300.56,True,82.666667,False,False,False,NaN,NaN,NaN,ACCOUNT_NOT_ACTIVE



Vista previa de review_queue (solo HOLD):


,request_id,decision,reason_code,severity,created_at,client_id,account_id,amount,currency,destination_type,...,settled_cash_after_request,dest_is_whitelisted,dest_days_since_change,dest_changed_recently_flag,urgent_risk_flag,duplicate_candidate_flag,duplicate_pair_request_id,mins_from_prev,mins_to_next,triggered_rules
0,W100064,HOLD,DUPLICATE_REQUEST,70.0,2026-03-05 06:46:35.358174+00:00,C007,A0032,4482.85,USD,crypto,...,7201.93,True,252.250000,False,False,True,W100064_DUP,NaN,7.0,DUPLICATE_REQUEST
1,W100064_DUP,HOLD,DUPLICATE_REQUEST,70.0,2026-03-05 06:53:35.358174+00:00,C007,A0032,4482.85,USD,crypto,...,7201.93,True,252.250000,False,False,True,W100064,7.0,NaN,DUPLICATE_REQUEST
2,W100044,HOLD,DUPLICATE_REQUEST,70.0,2026-03-05 09:08:16.872327+00:00,C020,A0003,10244.98,USD,bank,...,16389.82,True,4.666667,True,False,True,W100044_DUP,NaN,9.0,DUPLICATE_REQUEST | DEST_CHANGED_RECENTLY
3,W100044_DUP,HOLD,DUPLICATE_REQUEST,70.0,2026-03-05 09:17:16.872327+00:00,C020,A0003,10244.98,USD,bank,...,16389.82,True,4.666667,True,False,True,W100044,9.0,NaN,DUPLICATE_REQUEST | DEST_CHANGED_RECENTLY
4,W100004,HOLD,DUPLICATE_REQUEST,70.0,2026-03-05 09:19:55.351331+00:00,C020,A0026,21878.31,USD,bank,...,42612.78,True,4.666667,True,False,True,W100004_DUP,NaN,11.0,DUPLICATE_REQUEST | DEST_CHANGED_RECENTLY
5,W100010,HOLD,DUPLICATE_REQUEST,70.0,2026-03-05 09:28:21.519557+00:00,C017,A0034,11003.28,USD,bank,...,28859.19,True,195.333333,False,False,True,W100010_DUP,NaN,7.0,DUPLICATE_REQUEST
6,W100004_DUP,HOLD,DUPLICATE_REQUEST,70.0,2026-03-05 09:30:55.351331+00:00,C020,A0026,21878.31,USD,bank,...,42612.78,True,4.666667,True,False,True,W100004,11.0,NaN,DUPLICATE_REQUEST | DEST_CHANGED_RECENTLY
7,W100010_DUP,HOLD,DUPLICATE_REQUEST,70.0,2026-03-05 09:35:21.519557+00:00,C017,A0034,11003.28,USD,bank,...,28859.19,True,195.333333,False,False,True,W100010,7.0,NaN,DUPLICATE_REQUEST
8,W100073,HOLD,DUPLICATE_REQUEST,70.0,2026-03-05 17:49:45.791614+00:00,C024,A0025,3063.75,USD,bank,...,8537.65,True,23.000000,False,False,True,W100073_DUP,NaN,6.0,DUPLICATE_REQUEST
9,W100073_DUP,HOLD,DUPLICATE_REQUEST,70.0,2026-03-05 17:55:45.791614+00:00,C024,A0025,3063.75,USD,bank,...,8537.65,True,23.000000,False,False,True,W100073,6.0,NaN,DUPLICATE_REQUEST



Archivos generados:
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_rule_hits.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_final_decisions.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_review_queue.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_decision_summary.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_decision_counts.csv


## Ejercicio 3 – Resultados finales del caso de automatización

A partir de la base consolidada de solicitudes de retiro y de las reglas definidas en el enunciado, se construyó un motor de decisión para clasificar cada `request_id` en una de tres categorías:

- `APPROVE`
- `HOLD`
- `REJECT`

La lógica aplicada respetó las prioridades del ejercicio:

- las reglas de `REJECT` tienen prioridad sobre las de `HOLD`;
- las reglas de `HOLD` tienen prioridad sobre `APPROVE`;
- cuando una solicitud activa múltiples reglas, se asigna como `reason_code` la regla de mayor severidad.

## Reglas de rechazo aplicadas
- `KYC_NOT_VERIFIED`
- `ACCOUNT_NOT_ACTIVE`
- `UNWHITELISTED_HIGH_AML`
- `INVALID_AMOUNT`

## Reglas de revisión manual aplicadas
- `DUPLICATE_REQUEST`
- `INSUFFICIENT_SETTLED_AFTER_BUFFER`
- `INSUFFICIENT_AVAILABLE_AFTER_BUFFER`
- `DEST_CHANGED_RECENTLY`
- `URGENT_RISK_TIER`

## Resultado agregado
El motor clasificó las 128 solicitudes de la siguiente manera:

- **APPROVE:** 51
- **HOLD:** 49
- **REJECT:** 28

## Interpretación
Los rechazos estuvieron explicados principalmente por cuentas no activas y por KYC no verificado, lo cual es consistente con una política operativa conservadora.

Los casos en revisión manual se concentraron en:
- solicitudes potencialmente duplicadas;
- cambios recientes de destino;
- y retiros que comprometían el efectivo liquidado después de aplicar el buffer requerido.

El archivo de revisión se generó con únicamente las solicitudes `HOLD`, ordenadas de mayor a menor severidad, tal como exige el enunciado.

In [3]:
from pathlib import Path
import pandas as pd

# ============================================================
# EJERCICIO 3
# EXPORTACION FINAL DE ENTREGA
# ============================================================

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    required_dirs = {"data", "docs", "notebooks", "outputs", "src"}

    for candidate in candidates:
        if candidate.exists():
            dir_names = {p.name for p in candidate.iterdir() if p.is_dir()}
            if required_dirs.issubset(dir_names):
                return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. "
        "Ejecuta este notebook dentro de ~/Documents/prueba_insights"
    )

ROOT = find_project_root()
DOCS_DIR = ROOT / "docs"
EJ3_OUTPUT_DIR = ROOT / "outputs" / "ejercicio_3"

DOCS_DIR.mkdir(parents=True, exist_ok=True)
EJ3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

final_decisions_path = EJ3_OUTPUT_DIR / "ejercicio_3_final_decisions.csv"
review_queue_path = EJ3_OUTPUT_DIR / "ejercicio_3_review_queue.csv"
decision_counts_path = EJ3_OUTPUT_DIR / "ejercicio_3_decision_counts.csv"
decision_summary_path = EJ3_OUTPUT_DIR / "ejercicio_3_decision_summary.csv"

for path in [final_decisions_path, review_queue_path, decision_counts_path, decision_summary_path]:
    if not path.exists():
        raise FileNotFoundError(f"No existe el archivo requerido: {path}")

final_decisions = pd.read_csv(final_decisions_path)
review_queue = pd.read_csv(review_queue_path)
decision_counts = pd.read_csv(decision_counts_path)
decision_summary = pd.read_csv(decision_summary_path)

# ============================================================
# ARCHIVO PRINCIPAL DE ENTREGA
# ============================================================

submission_main = final_decisions[
    ["request_id", "decision", "reason_code", "severity"]
].copy()

# ============================================================
# ARCHIVO DE REVIEW DE ENTREGA
# Solo HOLD y con columnas operativas utiles
# ============================================================

submission_review = review_queue[
    [
        "request_id",
        "reason_code",
        "severity",
        "created_at",
        "client_id",
        "account_id",
        "amount",
        "currency",
        "destination_id",
        "requested_speed",
        "channel",
        "triggered_rules",
    ]
].copy()

# validacion
if not review_queue["decision"].eq("HOLD").all():
    raise ValueError("review_queue contiene decisiones distintas de HOLD")

# ============================================================
# RESUMEN EJECUTIVO
# ============================================================

decision_count_map = dict(zip(decision_counts["decision"], decision_counts["request_count"]))

approve_n = int(decision_count_map.get("APPROVE", 0))
hold_n = int(decision_count_map.get("HOLD", 0))
reject_n = int(decision_count_map.get("REJECT", 0))

summary_text = f"""
Ejercicio 3 - Caso de automatizacion

Resultado final:
- APPROVE: {approve_n}
- HOLD: {hold_n}
- REJECT: {reject_n}

Criterio aplicado:
- REJECT domina sobre HOLD
- HOLD domina sobre APPROVE
- Cuando una solicitud activa multiples reglas, se asigna el reason_code de mayor severidad

Observaciones:
- Los rechazos se explican principalmente por ACCOUNT_NOT_ACTIVE y KYC_NOT_VERIFIED.
- Los casos HOLD se concentran en DUPLICATE_REQUEST, DEST_CHANGED_RECENTLY e INSUFFICIENT_SETTLED_AFTER_BUFFER.
- El archivo de review contiene unicamente solicitudes HOLD y se encuentra ordenado por severidad descendente.
""".strip()

# ============================================================
# EXPORTACION FINAL
# ============================================================

submission_main.to_csv(EJ3_OUTPUT_DIR / "ejercicio_3_submission_main.csv", index=False)
submission_review.to_csv(EJ3_OUTPUT_DIR / "ejercicio_3_submission_review.csv", index=False)

with open(EJ3_OUTPUT_DIR / "ejercicio_3_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)

with open(DOCS_DIR / "ejercicio_3_resultados_finales.md", "w", encoding="utf-8") as f:
    f.write(
f"""# Ejercicio 3 - Caso de automatización
**Prueba técnica Insights**  
**Santiago Osorio Gómez**

## Resultado agregado
- **APPROVE:** {approve_n}
- **HOLD:** {hold_n}
- **REJECT:** {reject_n}

## Criterio aplicado
- `REJECT` domina sobre `HOLD`
- `HOLD` domina sobre `APPROVE`
- si una solicitud activa múltiples reglas, se asigna el `reason_code` de mayor severidad

## Lectura de resultados
Los rechazos estuvieron explicados principalmente por cuentas no activas y por KYC no verificado.

Los casos de revisión manual se concentraron principalmente en:
- solicitudes potencialmente duplicadas,
- cambios recientes de destino,
- y solicitudes que comprometían el efectivo liquidado después del buffer operativo.

El archivo de review se generó únicamente con solicitudes `HOLD`, ordenadas de mayor a menor severidad.
"""
    )

print(summary_text)

print("\nArchivos finales generados:")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_submission_main.csv'}")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_submission_review.csv'}")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_summary.txt'}")
print(f"- {DOCS_DIR / 'ejercicio_3_resultados_finales.md'}")

Ejercicio 3 - Caso de automatizacion

Resultado final:
- APPROVE: 51
- HOLD: 49
- REJECT: 28

Criterio aplicado:
- REJECT domina sobre HOLD
- HOLD domina sobre APPROVE
- Cuando una solicitud activa multiples reglas, se asigna el reason_code de mayor severidad

Observaciones:
- Los rechazos se explican principalmente por ACCOUNT_NOT_ACTIVE y KYC_NOT_VERIFIED.
- Los casos HOLD se concentran en DUPLICATE_REQUEST, DEST_CHANGED_RECENTLY e INSUFFICIENT_SETTLED_AFTER_BUFFER.
- El archivo de review contiene unicamente solicitudes HOLD y se encuentra ordenado por severidad descendente.

Archivos finales generados:
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_submission_main.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_submission_review.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_summary.txt
- /Users/santiago.os/Documents/prueba_insights/docs/ejercicio_3_resultados_finales.md


## Ejercicio 3 – Resultados finales del caso de automatización

A partir de la base consolidada de solicitudes de retiro y de las reglas definidas en el enunciado, se construyó un motor de decisión para clasificar cada `request_id` en una de tres categorías:

- `APPROVE`
- `HOLD`
- `REJECT`

La lógica aplicada respetó las prioridades del ejercicio:

- las reglas de `REJECT` tienen prioridad sobre las de `HOLD`;
- las reglas de `HOLD` tienen prioridad sobre `APPROVE`;
- cuando una solicitud activa múltiples reglas, se asigna como `reason_code` la regla de mayor severidad.

## Reglas de rechazo aplicadas
- `KYC_NOT_VERIFIED`
- `ACCOUNT_NOT_ACTIVE`
- `UNWHITELISTED_HIGH_AML`
- `INVALID_AMOUNT`

## Reglas de revisión manual aplicadas
- `DUPLICATE_REQUEST`
- `INSUFFICIENT_SETTLED_AFTER_BUFFER`
- `INSUFFICIENT_AVAILABLE_AFTER_BUFFER`
- `DEST_CHANGED_RECENTLY`
- `URGENT_RISK_TIER`

## Resultado agregado
El motor clasificó las 128 solicitudes de la siguiente manera:

- **APPROVE:** 51
- **HOLD:** 49
- **REJECT:** 28

## Interpretación
Los rechazos estuvieron explicados principalmente por cuentas no activas y por KYC no verificado, lo cual es consistente con una política operativa conservadora.

Los casos en revisión manual se concentraron en:
- solicitudes potencialmente duplicadas;
- cambios recientes de destino;
- y retiros que comprometían el efectivo liquidado después de aplicar el buffer requerido.

El archivo de revisión se generó con únicamente las solicitudes `HOLD`, ordenadas de mayor a menor severidad, tal como exige el enunciado.

In [4]:
from pathlib import Path
import pandas as pd

# ============================================================
# EJERCICIO 3
# EXPORTACION FINAL DE ENTREGA
# ============================================================

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    required_dirs = {"data", "docs", "notebooks", "outputs", "src"}

    for candidate in candidates:
        if candidate.exists():
            dir_names = {p.name for p in candidate.iterdir() if p.is_dir()}
            if required_dirs.issubset(dir_names):
                return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. "
        "Ejecuta este notebook dentro de ~/Documents/prueba_insights"
    )

ROOT = find_project_root()
DOCS_DIR = ROOT / "docs"
EJ3_OUTPUT_DIR = ROOT / "outputs" / "ejercicio_3"

DOCS_DIR.mkdir(parents=True, exist_ok=True)
EJ3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

final_decisions_path = EJ3_OUTPUT_DIR / "ejercicio_3_final_decisions.csv"
review_queue_path = EJ3_OUTPUT_DIR / "ejercicio_3_review_queue.csv"
decision_counts_path = EJ3_OUTPUT_DIR / "ejercicio_3_decision_counts.csv"
decision_summary_path = EJ3_OUTPUT_DIR / "ejercicio_3_decision_summary.csv"

for path in [final_decisions_path, review_queue_path, decision_counts_path, decision_summary_path]:
    if not path.exists():
        raise FileNotFoundError(f"No existe el archivo requerido: {path}")

final_decisions = pd.read_csv(final_decisions_path)
review_queue = pd.read_csv(review_queue_path)
decision_counts = pd.read_csv(decision_counts_path)
decision_summary = pd.read_csv(decision_summary_path)

# ============================================================
# ARCHIVO PRINCIPAL DE ENTREGA
# ============================================================

submission_main = final_decisions[
    ["request_id", "decision", "reason_code", "severity"]
].copy()

# ============================================================
# ARCHIVO DE REVIEW DE ENTREGA
# Solo HOLD y con columnas operativas utiles
# ============================================================

submission_review = review_queue[
    [
        "request_id",
        "reason_code",
        "severity",
        "created_at",
        "client_id",
        "account_id",
        "amount",
        "currency",
        "destination_id",
        "requested_speed",
        "channel",
        "triggered_rules",
    ]
].copy()

# validacion
if not review_queue["decision"].eq("HOLD").all():
    raise ValueError("review_queue contiene decisiones distintas de HOLD")

# ============================================================
# RESUMEN EJECUTIVO
# ============================================================

decision_count_map = dict(zip(decision_counts["decision"], decision_counts["request_count"]))

approve_n = int(decision_count_map.get("APPROVE", 0))
hold_n = int(decision_count_map.get("HOLD", 0))
reject_n = int(decision_count_map.get("REJECT", 0))

summary_text = f"""
Ejercicio 3 - Caso de automatizacion

Resultado final:
- APPROVE: {approve_n}
- HOLD: {hold_n}
- REJECT: {reject_n}

Criterio aplicado:
- REJECT domina sobre HOLD
- HOLD domina sobre APPROVE
- Cuando una solicitud activa multiples reglas, se asigna el reason_code de mayor severidad

Observaciones:
- Los rechazos se explican principalmente por ACCOUNT_NOT_ACTIVE y KYC_NOT_VERIFIED.
- Los casos HOLD se concentran en DUPLICATE_REQUEST, DEST_CHANGED_RECENTLY e INSUFFICIENT_SETTLED_AFTER_BUFFER.
- El archivo de review contiene unicamente solicitudes HOLD y se encuentra ordenado por severidad descendente.
""".strip()

# ============================================================
# EXPORTACION FINAL
# ============================================================

submission_main.to_csv(EJ3_OUTPUT_DIR / "ejercicio_3_submission_main.csv", index=False)
submission_review.to_csv(EJ3_OUTPUT_DIR / "ejercicio_3_submission_review.csv", index=False)

with open(EJ3_OUTPUT_DIR / "ejercicio_3_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)

with open(DOCS_DIR / "ejercicio_3_resultados_finales.md", "w", encoding="utf-8") as f:
    f.write(
f"""# Ejercicio 3 - Caso de automatización
**Prueba técnica Insights**  
**Santiago Osorio Gómez**

## Resultado agregado
- **APPROVE:** {approve_n}
- **HOLD:** {hold_n}
- **REJECT:** {reject_n}

## Criterio aplicado
- `REJECT` domina sobre `HOLD`
- `HOLD` domina sobre `APPROVE`
- si una solicitud activa múltiples reglas, se asigna el `reason_code` de mayor severidad

## Lectura de resultados
Los rechazos estuvieron explicados principalmente por cuentas no activas y por KYC no verificado.

Los casos de revisión manual se concentraron principalmente en:
- solicitudes potencialmente duplicadas,
- cambios recientes de destino,
- y solicitudes que comprometían el efectivo liquidado después del buffer operativo.

El archivo de review se generó únicamente con solicitudes `HOLD`, ordenadas de mayor a menor severidad.
"""
    )

print(summary_text)

print("\nArchivos finales generados:")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_submission_main.csv'}")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_submission_review.csv'}")
print(f"- {EJ3_OUTPUT_DIR / 'ejercicio_3_summary.txt'}")
print(f"- {DOCS_DIR / 'ejercicio_3_resultados_finales.md'}")

Ejercicio 3 - Caso de automatizacion

Resultado final:
- APPROVE: 51
- HOLD: 49
- REJECT: 28

Criterio aplicado:
- REJECT domina sobre HOLD
- HOLD domina sobre APPROVE
- Cuando una solicitud activa multiples reglas, se asigna el reason_code de mayor severidad

Observaciones:
- Los rechazos se explican principalmente por ACCOUNT_NOT_ACTIVE y KYC_NOT_VERIFIED.
- Los casos HOLD se concentran en DUPLICATE_REQUEST, DEST_CHANGED_RECENTLY e INSUFFICIENT_SETTLED_AFTER_BUFFER.
- El archivo de review contiene unicamente solicitudes HOLD y se encuentra ordenado por severidad descendente.

Archivos finales generados:
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_submission_main.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_submission_review.csv
- /Users/santiago.os/Documents/prueba_insights/outputs/ejercicio_3/ejercicio_3_summary.txt
- /Users/santiago.os/Documents/prueba_insights/docs/ejercicio_3_resultados_finales.md
